# 08 Graded positive control and a fair choice of the learning-rate factor (added after the protocol, review round 3)
The analysis of these runs is fixed in `protocols/ADDENDUM_2026-11-28.txt`, committed before the first run.

**Part 1, graded positive control (75 runs, plan A).** Three slots that pass only part of the code: `ma4` keeps 3 of the 4 values, `mb4` keeps 2 and `mc4` keeps 1; the other values are set to zero. With Id4 (all 4 values) and Z4 (none, notebook 07) they form a dose curve. It shows how small a loss this design detects.

**Part 2, factor reselection (90 short runs).** The factor k of Q4, Qp4, S4, Tr4 and Cl4 is chosen again from 3 seeds (100, 101, 102) and 2 folds (0 and 2), each with its own training and validation images; test images of those folds are not used. For each block, k is the value with the lowest mean best validation loss over the 6 runs.

**Part 3, retraining at the new factors (0 to 125 runs, plan A).** A block is retrained in all 25 runs only if its new k has not been run before (existing: Q4 100; Qp4 1 and 100; S4 100; Tr4 10 and 100; Cl4 100).

Outputs go to `runs_rev3/` only. **T4 GPU only.** Two sessions can run together: set `SESSION = 'B'` in the second one. Session A starts with part 2, session B with part 1. Do not change `CFG`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time
ROOT = '/content/drive/MyDrive/BUSI_QB'
CODE = os.path.join(ROOT, 'code'); PREP = os.path.join(ROOT, 'prepared')
RUNS = os.path.join(ROOT, 'runs_rev3'); os.makedirs(RUNS, exist_ok=True)
sys.path.insert(0, os.path.join(CODE, 'src'))
SESSION = 'A'                                     # 'B' in a second session
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
import qsim, qbus, runner
print(qbus.env_info())
assert 'T4' in str(qbus.env_info().get('gpu')), 'select a T4 GPU runtime'
assert qbus.CFG['min_epochs'] == 30 and qbus.CFG['max_epochs'] == 200 and qbus.CFG['es_patience'] == 20, qbus.CFG
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)

# masked slots: keep the first m of the 4 code values, set the others to zero (no weights; not named 'block')
KEEP = {'ma': 3, 'mb': 2, 'mc': 1}
_orig_make_block = qbus.make_block
def _make_block(kind, n, k):
    if kind in KEEP:
        mask = tf.constant([1.0] * KEEP[kind] + [0.0] * (n - KEEP[kind]), dtype=tf.float32)
        return keras.layers.Lambda(lambda t: t * mask, name='mask_block')
    return _orig_make_block(kind, n, k)
qbus.make_block = _make_block
for b in KEEP:
    if b not in qbus.BLOCKS:
        qbus.BLOCKS = qbus.BLOCKS + (b,)

# smoke check (no training): shared initial weights and the right number of zero values
for name, m_keep in (('ma4', 3), ('mb4', 2), ('mc4', 1)):
    m = qbus.build_model(name, k=1)
    d = qbus.apply_init(m, qbus.reference_init(4, 0))
    c = keras.Model(m.input, m.get_layer('mask_block').output)(np.random.rand(2, 256, 256, 1).astype('float32')).numpy()
    assert np.all(c[:, m_keep:] == 0) and np.all(np.abs(c[:, :m_keep]) > 0), (name, c)
    print(name, 'init diff', d, 'ok')
    del m
runner.free_memory()

F1 = ('q4', 'qp4', 's4', 'tr4', 'cl4')
EXISTING = {'q4': {100}, 'qp4': {1, 100}, 's4': {100}, 'tr4': {10, 100}, 'cl4': {100}}
sel  = [dict(plan='SEL', model=m, fold=f, seed=s, frac=1.0, k=k) for s in (100, 101, 102) for f in (0, 2) for m in F1 for k in (1, 10, 100)]
grad = [dict(plan='A', model=m, fold=f, seed=s, frac=1.0, k=1) for s in runner.SEEDS for f in range(5) for m in ('ma4', 'mb4', 'mc4')]
todo = sel + grad          # session A from the start (selection first), session B from the end (graded control first)
print(len(sel), 'selection runs,', len(grad), 'graded-control runs,',
      len(glob.glob(os.path.join(RUNS, '*', '*', 'summary.json'))), 'done')

In [ ]:
# 2. PARTS 1 AND 2 ----------------------------------------------------------------------------------------
def go(r):
    root = os.path.join(RUNS, r['plan']); os.makedirs(root, exist_ok=True)
    rid = runner.run_id(r)
    if r['plan'] == 'A' and r['model'] in F1:
        rid = rid + f"_k{r['k']}"
    if os.path.exists(os.path.join(root, rid, 'summary.json')):
        return
    if runner.claim(root, rid, SESSION):
        print(time.strftime('%m-%d %H:%M'), 'start', rid)
        if r['plan'] == 'A' and r['model'] in F1:      # part 3: keep the factor in the folder name
            s = runner.one_run(dict(r, plan='A'), PREP, os.path.join(root, '_tmp_' + rid), session=SESSION)
            os.replace(os.path.join(root, '_tmp_' + rid, runner.run_id(r)), os.path.join(root, rid))
            os.rmdir(os.path.join(root, '_tmp_' + rid))
        else:
            s = runner.one_run(r, PREP, root, session=SESSION)
        if r['plan'] == 'A':
            assert s['epochs'] >= 30, ('stopped before 30 epochs', rid)
            print('   Dice', round(s['test_dice'], 4), 'epochs', s['epochs'], 'min', round(s['minutes'], 1))
        else:
            print('   best val loss', round(s['best_val_loss'], 4), 'epochs', s['epochs'], 'min', round(s['minutes'], 1))

for r in runner.ordered(todo, SESSION):
    go(r)

In [ ]:
# 3. NEW FACTORS AND PART 3 (run when all 90 selection runs are done) ------------------------------------
rows = []
for r in sel:
    p = os.path.join(RUNS, 'SEL', runner.run_id(r), 'summary.json')
    if os.path.exists(p):
        s = json.load(open(p)); rows.append(dict(model=s['model'], k=s['k'], fold=s['fold'], seed=s['seed'], loss=s['best_val_loss']))
t = pd.DataFrame(rows)
assert len(t) == 90, f'{len(t)} of 90 selection runs done; wait for the rest'
mean = t.groupby(['model', 'k']).loss.mean().unstack()
wins = t.loc[t.groupby(['model', 'fold', 'seed']).loss.idxmin()].groupby(['model', 'k']).size().unstack(fill_value=0)
newk = mean.idxmin(axis=1).astype(int).to_dict()
print(mean.round(4)); print('runs won by each k'); print(wins); print('new factors', newk)
os.makedirs(os.path.join(ROOT, 'analysis', 'rev3'), exist_ok=True)
t.to_csv(os.path.join(ROOT, 'analysis', 'rev3', 'selection_runs.csv'), index=False)
json.dump(newk, open(os.path.join(ROOT, 'analysis', 'rev3', 'new_factors.json'), 'w'))
part3 = [dict(plan='A', model=m, fold=f, seed=s, frac=1.0, k=newk[m]) for m in F1 if newk[m] not in EXISTING[m]
         for s in runner.SEEDS for f in range(5)]
print(len(part3), 'runs to retrain:', sorted({(r['model'], r['k']) for r in part3}))
for r in runner.ordered(part3, SESSION):
    go(r)

In [ ]:
# 4. COLLECT (when parts 1 to 3 are done) ----------------------------------------------------------------
newk = json.load(open(os.path.join(ROOT, 'analysis', 'rev3', 'new_factors.json')))
N = 75 + 25 * sum(1 for m in F1 if newk[m] not in EXISTING[m])
dirs = sorted(d for d in glob.glob(os.path.join(RUNS, 'A', '*')) if os.path.isdir(d) and not os.path.basename(d).startswith('_tmp_'))
res = []
for d in dirs:
    s = os.path.join(d, 'summary.json')
    if not os.path.exists(s):
        continue
    out = dict(summary=json.load(open(s)))
    out['summary']['run_id'] = os.path.basename(d)
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement'):
        f = os.path.join(d, nm + '.csv')
        if os.path.exists(f):
            t = pd.read_csv(f); t.insert(0, 'run_id', out['summary']['run_id']); out[nm] = t
    res.append(out)
print(f'done {len(res)} / {N}, running {len(glob.glob(os.path.join(RUNS, "*", "*.lock")))}')
if len(res) == N:
    OUT = os.path.join(ROOT, 'analysis', 'rev3'); os.makedirs(OUT, exist_ok=True)
    summ = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in r['summary'].items()} for r in res])
    assert (summ.epochs >= 30).all(), summ.loc[summ.epochs < 30, 'run_id'].tolist()
    summ.to_csv(os.path.join(OUT, 'summaries.csv'), index=False)
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi', 'code_replacement'):
        parts = [r[nm] for r in res if nm in r]
        if parts:
            t = pd.concat(parts, ignore_index=True)
            for c in t.columns:
                if t[c].dtype == float:
                    t[c] = t[c].round(5)
            t.to_csv(os.path.join(OUT, nm + '.csv.gz'), index=False, compression='gzip')
            print(nm, len(t), 'rows')
    print(summ.groupby(['model', 'k']).agg(runs=('run_id', 'size'), dice=('test_dice', 'mean'), epochs_min=('epochs', 'min')).round(4))
    print('Files written to', OUT)